# Eksperimentere med Neo4j
Sjekke at vi kan kjøre Neo4J

Starte med
```
sudo apt install podman
```
Her kjører vi på egne maskiner heller enn å benytte PITs test-server.  Årsaken er enkel: Fungerer uten VDI.

## Installere Neo4J

In [83]:
%%bash
PWD=$(pwd)
mkdir -p neo4j
mkdir -p neo4j/data
mkdir -p neo4j/logspo
mkdir -p neo4j/plugins
mkdir -p neo4j/import

# Dette laster ned to plugins fra Neo4j (som må importeres for hånd om vi skal kjøre på VDI)

podman run \
    -p 7474:7474 -p 7687:7687 \
    --userns=keep-id \
    -e NEO4J_PLUGINS='["apoc", "graph-data-science"]' \
    -e NEO4J_dbms_security_procedures_unrestricted='gds.*,apoc.*' \
    -e NEO4J_dbms_security_procedures_allowlist='gds.*,apoc.*' \
    -e NEO4J_apoc_import_file_enabled=true \
    -v $PWD/neo4j/data:/data:Z \
    -v $PWD/neo4j/logs:/logs:Z \
    -v $PWD/neo4j/import:/import:Z \
    -v $PWD/neo4j/plugins:/plugins:Z \
    -e NEO4J_AUTH=neo4j/password \
    -d docker.io/library/neo4j:latest

2e8f4fd931c33ec849642cbe21022fd9ead6db3babc69de476432e097073b0d1


Når man er ferdig er det bare å kopiere identifikatoren over inn i neste kall

In [163]:
%%bash
podman kill d1349c6d65b5e1ac85ac00826fbe43f4e4e51a6553dd897b456c304a2d69734d

d1349c6d65b5e1ac85ac00826fbe43f4e4e51a6553dd897b456c304a2d69734d


Gi databasen litt tid til å starte

In [146]:
from neo4j import GraphDatabase

# 1. Define connection details
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Connection Successful!")

Connection Successful!


In [147]:
# Vår sikker på at vi ikke starter med gamle data
records, summary, keys = driver.execute_query(
    """match (n) detach delete n""")
#
for s in summary.gql_status_objects:
    print(s)
#

note: successful completion - omitted result


(Den tomme) grafen er nå tilgjengelig på 
```
http://localhost:7474/browser/
```


### Laste inn APOC
Sjekke at vi har APOC tilgjengelig

In [95]:
records, summary, keys = driver.execute_query(
    """RETURN apoc.version() AS version;""")

print(f"Server Address: {summary.server.address}")
print("Keys:")
for k in range(len(keys)):
    print(f"\t{keys[k]}: {records[k]}")
#

print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")



Server Address: 127.0.0.1:7687
Keys:
	version: <Record version='2025.11.2'>
Grafen
	Nye noder: 0
	Nye kanter: 0
Ressursbruk
	Kjøringen: 2ms
	Å konsumere: 1ms


### Laste inn GDS
Har vi GDS?

In [96]:
records, summary, keys = driver.execute_query(
    """
    CALL gds.version();
    """)

print("Keys:")
for k in range(len(keys)):
    print(f"\t{keys[k]}: {records[k]}")
#

Keys:
	gdsVersion: <Record gdsVersion='2.24.0'>


## Fra Networkx til Neo4j
### Eksportere fra Networkx

In [148]:
import gzip
import networkx as nx

with gzip.open("data/email.edgelist.txt.gz", "rt") as fd:
    G = nx.read_edgelist(fd, create_using=nx.DiGraph())
#
G.remove_edges_from(nx.selfloop_edges(G))
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

nx.set_node_attributes(G, ":Person", name="labels")
nx.set_edge_attributes(G, "EPOST", name="label")

# Når vi eksporterer, da gis Networkx' identifikator (det vil si 6 i G["6"]) navnet id.
#    <node id="63">
#       <data key="labels">:Person</data>
#    </node>
# Vi skal senere importere i neo4j, og der er id et "belastet" navn, i betydningen at det var en intern identifikator.
# "id" har i Neo4J versjon 5 blitt oppdatert til elementId.
for n, d in G.nodes(data=True):
    # n er str
    G.nodes[n]["Navn"] = "Bruker " + n
#

# Skriv ut grafen
nx.write_graphml(G, "neo4j/import/large_graph.graphml", named_key_ids=True)
print("ok")

Nodes: 57194
Edges: 103083
ok


### Lese inn i Neo4j

In [149]:
records, summary, keys = driver.execute_query(
    """CALL apoc.import.graphml("large_graph.graphml", {storeNodeIds: true, readLabels: true})""")
# for enkelt å pakke opp svaret
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"\t{k}: {record_dict[k]}")
#
print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")
print("Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder")
      
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	file: large_graph.graphml
	source: file
	format: graphml
	nodes: 57194
	relationships: 103083
	properties: 57194
	time: -248
	rows: 0
	batchSize: -1
	batches: 0
	done: True
	data: None
Grafen
	Nye noder: 0
	Nye kanter: 0
Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder
Ressursbruk
	Kjøringen: 83ms
	Å konsumere: -241ms


#### Fjerne id

In [150]:
# NetworkX lager en "id" på hver node.  Det vil vi ikke ha, for "id" er en belastet attribut i Neo4J.
records, summary, keys = driver.execute_query(
    """MATCH (n:Person)
    WHERE n.id IS NOT NULL
    REMOVE n.id
    RETURN COUNT (n)""")
#
print(f"Fjernet: {summary.counters.properties_set}") # "set" betyr her "modifisert"
for s in summary.gql_status_objects:
    print(s)
#
for r in records:
    print(r)

Fjernet: 57194
note: successful completion
<Record COUNT (n)=57194>


## Ting og tang

### Sette en index

In [151]:
records, summary, keys = driver.execute_query(
    """
    CREATE INDEX
    IF NOT EXISTS 
    FOR (n:Person) ON (n.Navn);
    """)      
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print("ok")

Ressursbruk
	Kjøringen: 12ms
ok


### Hente data
Sjekke noen velkjente noder:
- Bruker 23
- Bruker 26
- Bruker 15
- Bruker 6

Den "andre" måten å bruke driveren på, er gjennom sesjoner.

Resultatet må hentes i konteksten

In [152]:
with driver.session(database="neo4j") as session:
    resultat = session.run("""
    MATCH (p:Person)
    WHERE p.Navn IN ["Bruker 23", "Bruker 26", "Bruker 15", "Bruker 6"]
    RETURN p.Navn AS navn, COUNT {(p)--()} as naboer
    """)
    det_hele = []
    for r in resultat:
        det_hele += [r]
    #
    for s in resultat.consume().gql_status_objects:
        print(f"Hvordan gikk det: {s}")
    #
#
for d in det_hele:
    print(d)


Hvordan gikk det: note: successful completion
<Record navn='Bruker 23' naboer=137>
<Record navn='Bruker 26' naboer=99>
<Record navn='Bruker 15' naboer=75>
<Record navn='Bruker 6' naboer=263>


In [153]:
# Legg inn
records, summary, keys = driver.execute_query(
    """
    MATCH (n:Person)
    SET n.antallKanter = COUNT { (n)--() }
    """)
for r in records:
    # Skal være tom
    innhold = r.data()
    print(innhold)
#
records, summary, keys = driver.execute_query(
    """
    CREATE INDEX
    IF NOT EXISTS 
    FOR (n:Person) ON (n.antallKanter);
    """)
for r in records:
    # Skal være tom
    innhold = r.data()
    print(innhold)
#
print("ok")

ok


In [154]:
records, summary, keys = driver.execute_query(
    """MATCH p=()-->(:Person {Navn:"Bruker 6"}) RETURN p;
""")
# Returnerer et sett av STIER som ender i noden identifisert med id=6
for r in records:
    innhold = r.data()
    print(innhold)
    break # Første er tilstrekkelig
#
records, summary, keys = driver.execute_query(
    """MATCH (p)-->(:Person {Navn:"Bruker 6"}) RETURN p;
""")
# Returnerer et sett av NODER som har relasjoner til noden identifisert med id=6
for r in records:
    innhold = r.data()
    print(innhold)
    break # Første er nok
#
    

{'p': [{'antallKanter': 1, 'Navn': 'Bruker 24805'}, 'EPOST', {'antallKanter': 263, 'Navn': 'Bruker 6'}]}
{'p': {'antallKanter': 1, 'Navn': 'Bruker 24805'}}


## Bruke GDS

Cypher egner seg til "enkle ting".  Det vil si transaksjoner på og med noder, men er ikke egnet til å implementere algoritmer.  GDS er imkplementert i Java og kjører inne i Neo4J og har fri tilgang til alle interne datastrukturer.

Fordi mange graf-algoritmer i praksis bruker alle noder i grafen, er valget med å kjøre i en definert sub-graf og kreve at den er i hukommelsen, et design som gir mening.  Tross alt er tilgang til data i hukommelsen minst fire størrelsesordner raskere, og ett eneste søk på disk kan ødelegge alt.  For å gjøre dette mulig er flyten delt i tre, og eksplisitt funksjonalitet for testing tilgjengelig.

Flytens tre (fire) steg er:
- Konstruere (sub)grafen i hukommelsen ved å velge noder og relasjoner som er relevante.  Det gjøres enten med primitiver tilgjengelig i GDS dersom subgrafen skal bestå av  "enkle ting".  Eller brukes Cyhper til å velge ut noder og relasjoner;
- For sikkerhets skyld bør man be om et estimat på algoritmen som skal kjøres.  Estimatet gjøres ved å estimere algoritmen opp mot subgrafen som er laget;
- Kjøre algoritmen, og
- Fjerne subgrafen når det ikke lenger er brhov for den.

Algoritmen bør ha noen sideeffekter.  Det er fire måter å skape dem:
- **stream**: Data returneres til kalleren, som enten er Python-koden eller i nettleseren;
- **stats**: Returnerer statestikk (metainformasjon) heller enn noder og relasjoner;
- **mutate**: Skriver endringer tilbake til subgrafen i hukommelsen, og
- **write**: Skriver endringer tilbake i selve databasen.




### Eksempel på lasting med merkede noder

Det enkleste er å merke det man vil ha med, og så laste dem inn direkte.  Vi har allerede satt merkelappen `antallKanter` på hver node, (og satt på en index) så vi laster med dem.



#### Merke nodene

In [189]:
records, summary, keys = driver.execute_query(
    """
    MATCH (p:Person)
    WHERE p.antallKanter > 100
    SET p:Viktig;
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print("ok") 

Hvordan gikk det: note: successful completion
Ressursbruk
	Kjøringen: 21ms
ok


#### Lage subgrafen
Så laster vi de nodene sammen med (kun) `EPOST`-relasjonene for det kan jo være andre relasjoner i databasen:

In [181]:
records, summary, keys = driver.execute_query(
    """
    CALL gds.graph.project(
        'ViktigGraf',
        ['Viktig'],    // Første "søk": Noder
        ['EPOST']      // Andre  "søk": Kanter
    )
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print("ok") 


Hvordan gikk det: note: successful completion
Ressursbruk
	Kjøringen: 0ms
ok


#### Et estimat på kjøringen

In [165]:
# Et estimat
records, summary, keys = driver.execute_query(
    """
    CALL gds.pageRank.stream.estimate (  // Få estimat på å sende resultatene tilbake til meg (stream)
        'ViktigGraf',
        {
        maxIterations: 20,   // Parametre for algoritmen
        dampingFactor: 0.85
        }
        )
    YIELD nodeCount, bytesMin, bytesMax, requiredMemory
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
for r in records:
    innhold = r.data()
    print(innhold)
    break # Første er nok
#
print("ok") 


Hvordan gikk det: note: successful completion
Ressursbruk
	Kjøringen: 1ms
{'nodeCount': 209, 'bytesMin': 5872, 'bytesMax': 5872, 'requiredMemory': '5872 Bytes'}
ok


#### Hente data fra subgrafen

In [168]:
# La oss få data
records, summary, keys = driver.execute_query(
    """
    CALL gds.pageRank.stream('ViktigGraf')
    YIELD nodeId, score
    RETURN gds.util.asNode(nodeId).Navn AS navn, score
    ORDER BY score DESC
    LIMIT 5
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
for r in records:
    innhold = r.data()
    print(innhold)
#
print("ok") 


Hvordan gikk det: note: successful completion
Ressursbruk
	Kjøringen: 0ms
{'navn': 'Bruker 11798', 'score': 8.272769620366098}
{'navn': 'Bruker 12586', 'score': 2.593843757509939}
{'navn': 'Bruker 14603', 'score': 2.475295656143867}
{'navn': 'Bruker 880', 'score': 1.6377652947797572}
{'navn': 'Bruker 737', 'score': 1.6068969822908044}
ok


#### Fjerne subgrafen

In [184]:
# Fjerne subgrafen
records, summary, keys = driver.execute_query(
    """
    CALL gds.graph.drop('ViktigGraf', false)  // false = Ikke få feil om den ikke finnes
    YIELD graphName
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
if not records:
    print("Allerede slettet")
else:
    print(f"Navnet på subgrafen: {records}")
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

print("ok") 



Hvordan gikk det: note: successful completion
Allerede slettet
Ressursbruk
	Kjøringen: 1ms
ok


#### Fjerne merkelappene

In [190]:
# Til slutt,gjerne merkelappen

records, summary, keys = driver.execute_query(
    """
    MATCH (p:Viktig) 
    REMOVE p:Viktig
    RETURN COUNT (p);
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
for rec in records:
    for r in rec:
        print(f"Slettet: {r}")
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

print("ok") 

Hvordan gikk det: note: successful completion
Slettet: 209
Ressursbruk
	Kjøringen: 28ms
ok



### Eksempel uten merkelapp

Vi kan, muligens i tillegg, kjøre Cypher som en del av byggingen.  Vi trenger to (2) Cypher-kall: Ett for noder og ett for kanter.
```
CALL gds.graph.project(
   // Her finner vi hvilke noder som skal være med
  'ViktigGraf',            // Navnet på subgrafen
  'MATCH (p:Person) 
   WHERE COUNT { (p)-[:EMAIL]-() } > 100   // Bare noder med mer enn 100 eposter
   RETURN elementId(p) AS id',             // sendes til neste steg
   // Her finner vi hvilke kanter som skal være med.
   // Denne krever to tellinger for å sikre at det ikke forsøkes å lage en relasjon til noder
   // som ikke er med (har mindre enn 100 eposter).
  'MATCH (p1:Person)-[r:EMAIL]-(p2:Person) 
   WHERE COUNT { (p1)-[:EMAIL]-() } > 100  // 
     AND COUNT { (p2)-[:EMAIL]-() } > 100
   RETURN elementId(p1) AS source, elementId(p2) AS target, type(r) AS type' // Relationship query
)
```

### Eksempel på å skrive tilbake til databsen

Her ser vi hvordan data skrives tilbake til de nodene det gjelder

```
CALL gds.pageRank.write(
  'ViktigGraf',
  {
    writeProperty: 'HvorViktig' // Hva skal egenskapen på noden hete
  }
)
YIELD nodePropertiesWritten, ranIterations // Dette er returverdien og ikke sideeffekten!
```
Så kan vi hente resultatene med Cypher:
```
MATCH (p:Person)
WHERE p.HvorViktig IS NOT NULL
RETURN p.Navn, p.HvorViktig
ORDER BY p.HvorViktig DESC
LIMIT 10
```


### GDS eksempel
Først bruke Louven til å finne gjenger, og så finne den viktigste noden i hver gjeng.

In [ ]:
// Create a named in-memory graph called 'social-network'
// We project 'Person' nodes and 'KNOWS' relationships
CALL gds.graph.project(
  'social-network',
  'Person',
  {
    KNOWS: {
      orientation: 'UNDIRECTED' // Treat relationships as two-way for better clustering
    }
  }
)
YIELD graphName, nodeCount, relationshipCount;

In [ ]:
// Run Louvain to find clusters and save them to the in-memory graph
CALL gds.louvain.mutate('social-network', {
  mutateProperty: 'communityId'
})
YIELD communityCount, modularity;

In [ ]:
// Find the most influential person (PageRank) within each cluster
CALL gds.pageRank.stream('social-network')
YIELD nodeId, score
RETURN 
  gds.util.asNode(nodeId).name AS Name, 
  gds.util.asNode(nodeId).communityId AS Cluster, 
  score AS InfluenceScore
ORDER BY Cluster ASC, InfluenceScore DESC;

### Finne gjenger

Dette er hvordan Neo4J tegner grafen i en *browser*:
![Grafen](data/Neo4j-graf.png)
Vi skal forsøke å gjenskape dette bildet

// 1. Project the graph into memory
CALL gds.graph.project('clusterGraph', '*', '*')
YIELD graphName, nodeCount;

In [ ]:
// 2. Run Louvain and write the results back to the nodes
CALL gds.louvain.write('clusterGraph', {
  writeProperty: 'community'
})
YIELD communityCount, modularity;

In [ ]:
// Find communities with fewer than 10 nodes and unset their community property
MATCH (n)
WITH n.community AS comm, count(n) AS clusterSize
WHERE clusterSize < 10
MATCH (m {community: comm})
REMOVE m.community;

In [ ]:
MATCH (n) WHERE n.community IS NOT NULL RETURN n LIMIT 1000